# LlamaIndex PDF RAG Lab — Final Notebook

This notebook builds a **citation-grounded PDF QA system** over a small folder of PDFs using:

- **LlamaIndex**
- **BGE embeddings**: `BAAI/bge-base-en-v1.5`
- **Local LLM**: `Qwen/Qwen3.5-9B`
- **CitationQueryEngine** for grounded answers with sources

## Pipeline

**PDFs → Documents → metadata → cleaned text → nodes/chunks → embeddings → index → retrieval → LLM answer → citations**

## Expected folder structure

```text
data/pdfs/
├── worldbank_ai_2025.pdf
├── worldbank_dpi_2025.pdf
├── oecd_gov_ai_2025.pdf
└── oecd_ai_openness_2025.pdf
```

## Notes

- This notebook keeps the pipeline transparent: we inspect documents, nodes, retrieval results, and cited answers.
- If PDF text extraction still looks messy for some files, switch to a stronger parser later. For the teaching lab, this version adds a cleanup pass to reduce common Unicode / spacing issues.

In [ ]:
# Install / upgrade packages
# Qwen3.5 currently needs a recent transformers build.

%pip install -q -U llama-index llama-index-embeddings-huggingface llama-index-llms-huggingface
%pip install -q -U sentence-transformers accelerate bitsandbytes pypdf peft
%pip install -q -U "transformers[torch] @ git+https://github.com/huggingface/transformers.git@main"

print("Packages installed. Restart the kernel if this is your first run.")

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import re
import unicodedata
import textwrap

import torch
from transformers import BitsAndBytesConfig, AutoTokenizer

from llama_index.core import (
    Settings,
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
    set_global_tokenizer,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.query_engine import CitationQueryEngine
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM

## 2. Point to the PDF folder

In [ ]:
DATA_DIR = Path("data/pdfs")
PERSIST_DIR = Path("storage/pdf_rag_index")

assert DATA_DIR.exists(), f"Missing directory: {DATA_DIR}"
pdf_files = sorted(DATA_DIR.glob("*.pdf"))
assert pdf_files, f"No PDFs found in {DATA_DIR}"

print("PDF files found:")
for p in pdf_files:
    print("-", p.name)

## 3. File-level metadata

We attach simple metadata early so that retrieval results and citations are easier to interpret.

In [ ]:
def get_file_metadata(file_path: str):
    path = Path(file_path)
    stem = path.stem.lower()

    source_org = "unknown"
    if "worldbank" in stem:
        source_org = "World Bank"
    elif "oecd" in stem:
        source_org = "OECD"

    return {
        "file_name": path.name,
        "doc_id": path.stem,
        "source_org": source_org,
        "file_type": path.suffix.lower().replace(".", ""),
    }

## 4. Load PDFs into LlamaIndex `Document`s

In [ ]:
reader = SimpleDirectoryReader(
    input_dir=str(DATA_DIR),
    file_metadata=get_file_metadata,
    filename_as_id=True,
)

documents = reader.load_data()
print(f"Loaded {len(documents)} documents")

## 5. Clean extracted text

PDF extraction often introduces:
- ligatures and Unicode oddities
- non-breaking spaces
- zero-width spaces
- ugly line breaks / spacing

This lightweight cleanup helps the lab behave more consistently.

In [ ]:
def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00a0", " ")  # non-breaking space
    text = text.replace("\u200b", "")   # zero-width space
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

for doc in documents:
    doc.text = clean_text(doc.text)

## 6. Inspect loaded documents

In [ ]:
for i, doc in enumerate(documents):
    print(f"\n=== Document {i+1} ===")
    print("Doc ID:", doc.doc_id)
    print("Metadata:", doc.metadata)
    preview = doc.text.replace("\n", " ")
    print("Preview:")
    print(textwrap.shorten(preview, width=500, placeholder=" ..."))

## 7. Configure the embedding model

Final choice for the lab:

**`BAAI/bge-base-en-v1.5`**

This is a stable English retrieval model and integrates cleanly with LlamaIndex.

In [ ]:
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-base-en-v1.5"
)

Settings.embed_model = embed_model
print("Embedding model configured.")

## 8. Configure the local LLM

Final choice for the lab:

**`Qwen/Qwen3.5-9B`**

Notes:
- Qwen3.5 needs a recent `transformers`
- 4-bit quantization is used to reduce memory usage
- keep answers concise for a classroom demo

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-9B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

llm = HuggingFaceLLM(
    model_name=MODEL_NAME,
    tokenizer_name=MODEL_NAME,
    context_window=8192,
    max_new_tokens=384,
    model_kwargs={
        "quantization_config": quantization_config,
        "device_map": "auto",
    },
    generate_kwargs={
        "temperature": 0.7,
        "top_p": 0.8,
        "do_sample": True,
    },
)

Settings.llm = llm

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
set_global_tokenizer(tokenizer.encode)

print("LLM configured.")

## 9. Split documents into nodes

We use `SentenceSplitter` so chunking remains visible and controllable.

In [ ]:
splitter = SentenceSplitter(
    chunk_size=800,
    chunk_overlap=100,
)

nodes = splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} nodes.")

## 10. Inspect a few nodes

In [ ]:
for i, node in enumerate(nodes[:5]):
    print(f"\n=== Node {i+1} ===")
    print("Node ID:", node.node_id)
    print("Metadata:", node.metadata)
    preview = node.text.replace("\n", " ")
    print(textwrap.shorten(preview, width=500, placeholder=" ..."))

## 11. Build or load the vector index

To make the classroom demo faster, this notebook supports persistence.

- If a saved index exists, load it.
- Otherwise build it from the nodes and persist it.

In [ ]:
if PERSIST_DIR.exists():
    print(f"Loading existing index from: {PERSIST_DIR}")
    storage_context = StorageContext.from_defaults(persist_dir=str(PERSIST_DIR))
    index = load_index_from_storage(storage_context)
else:
    print("Building new VectorStoreIndex...")
    index = VectorStoreIndex(nodes)
    PERSIST_DIR.mkdir(parents=True, exist_ok=True)
    index.storage_context.persist(persist_dir=str(PERSIST_DIR))
    print(f"Index persisted to: {PERSIST_DIR}")

print("Index ready.")

## 12. Inspect retrieval before generation

Always inspect retrieved nodes before trusting the final answer.

In [ ]:
retriever = index.as_retriever(similarity_top_k=3)

query = "How do these reports connect AI governance, openness, and digital infrastructure?"
retrieved_nodes = retriever.retrieve(query)

print(f"Query: {query}")
print(f"Retrieved {len(retrieved_nodes)} nodes\n")

for i, n in enumerate(retrieved_nodes, 1):
    print(f"=== Rank {i} ===")
    print("Score:", getattr(n, "score", None))
    print("Metadata:", n.metadata)
    preview = n.text.replace("\n", " ")
    print(textwrap.shorten(preview, width=600, placeholder=" ..."))
    print("-" * 100)

## 13. Plain query engine

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=3)

response = query_engine.query(
    "Summarize how the documents discuss AI governance, openness, and digital infrastructure."
)

print(response)

## 14. Citation-grounded query engine

In [ ]:
citation_query_engine = CitationQueryEngine.from_args(
    index,
    similarity_top_k=3,
    citation_chunk_size=512,
)

citation_response = citation_query_engine.query(
    "Compare how the World Bank and OECD documents talk about AI readiness, governance, and policy design."
)

print("Answer:\n")
print(citation_response)

print("\nSources:\n")
for i, src in enumerate(citation_response.source_nodes, 1):
    print(f"=== Source {i} ===")
    print("Metadata:", src.node.metadata)
    preview = src.node.text.replace("\n", " ")
    print(textwrap.shorten(preview, width=600, placeholder=" ..."))
    print("-" * 100)

## 15. Try a few more queries

In [ ]:
sample_queries = [
    "What is digital public infrastructure?",
    "Which document focuses most directly on AI openness?",
    "What recurring themes appear across all four documents?",
    "How do the reports frame institutional readiness for AI?",
]

for q in sample_queries:
    print("\n" + "="*120)
    print("QUESTION:", q)
    ans = citation_query_engine.query(q)
    print("\nANSWER:")
    print(ans)

## 16. Negative test

A good RAG system should avoid inventing unsupported answers.

In [ ]:
bad_query = "Which document recommends a specific GPU brand for AI deployment?"
bad_response = citation_query_engine.query(bad_query)

print("QUESTION:", bad_query)
print("\nANSWER:")
print(bad_response)

## 17. Small experiment: change `top_k`

This helps students feel how retrieval settings affect generation.

In [ ]:
for k in [2, 5]:
    qe = index.as_query_engine(similarity_top_k=k)
    resp = qe.query("What themes recur across the four documents?")
    print(f"\n{'='*80}\nTop-k = {k}\n")
    print(resp)

## 18. Optional reset

Use this only if you change:
- chunk size
- overlap
- embedding model
- corpus files

Then rebuild the index from scratch.

In [ ]:
# OPTIONAL: uncomment to force rebuild on the next run
# import shutil
# if PERSIST_DIR.exists():
#     shutil.rmtree(PERSIST_DIR)
#     print(f"Deleted persisted index: {PERSIST_DIR}")

## 19. Suggested discussion prompts for class

1. Did the retrieved nodes actually support the answer?
2. Were citations meaningful and easy to inspect?
3. How would chunk size changes affect retrieval?
4. What happens when the query is outside the corpus?
5. What part of the pipeline would you debug first if the answer looked wrong?

## Key takeaway

**Retrieval selects the evidence. The LLM synthesizes the answer. Citations make the answer inspectable.**